# Phase 4: Evaluate with EvalHub SDK + MLflow

This notebook runs LLM evaluations through the [EvalHub](https://github.com/eval-hub/eval-hub) REST API using the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) Python client. All results are automatically tracked in **MLflow**.

## Why use EvalHub?

| Feature | LMEvalJob (Phase 1-2) | EvalHub (Phase 4) |
|---------|----------------------|-------------------|
| Interface | Kubernetes CR (YAML) | Python SDK / REST API |
| Frameworks | lm-evaluation-harness only | lm-eval, RAGAS, LightEval, GuideLLM, ... |
| Multi-benchmark | One task per CR | Multiple benchmarks per request |
| Experiment tracking | Manual (Pod logs) | **Built-in MLflow** (metrics, params, artifacts) |
| Result management | `oc get lmevaljob` | Centralized API + MLflow UI |
| Job management | `oc delete lmevaljob` | SDK `client.jobs.cancel()` |

## Prerequisites

- **0_setup/0_model_deploy.ipynb** completed (model deployed)
- **0_setup/1_LMEval_setup.ipynb** completed (RBAC and secrets)
- **0_setup/2_eval_hub_setup.ipynb** completed (EvalHub SDK installed and verified)
- EvalHub service running on the cluster
- MLflow tracking server accessible from EvalHub

## Step 1: Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
EVALHUB_URL = os.getenv("EVALHUB_URL", "http://evalhub:8080")
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", None)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LIMIT = int(os.getenv("LIMIT", "5"))

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {BASE_URL}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

## Step 2: Initialize the EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

## Step 3: Discover Available Korean Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider and filter for Korean tasks.

In [ ]:
all_benchmarks = client.benchmarks.list()

korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "hrm8k"]
korean_benchmarks = [
    bm for bm in all_benchmarks.items
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Total benchmarks available: {all_benchmarks.total_count}")
print(f"Korean benchmarks found:    {len(korean_benchmarks)}")
print("=" * 70)
for bm in korean_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

---

## Evaluation 1: Single Korean Benchmark

Start with a single benchmark to verify the pipeline works end-to-end.

### 1-A: Submit the Job

In [ ]:
single_request = JobSubmissionRequest(
    name="kmmlu-law-eval",
    description="Korean MMLU Law benchmark - single task smoke test",
    tags=["korean", "kmmlu", "smoke-test"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-kmmlu-law",
        tags=[
            ExperimentTag(key="model_family", value="gemma-4"),
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="benchmark_suite", value="kmmlu"),
            ExperimentTag(key="evaluation_type", value="smoke-test"),
        ],
    ),
)

job = client.jobs.submit(single_request)

print(f"Job submitted successfully!")
print(f"  Job ID:        {job.id}")
print(f"  Name:          {job.name}")
print(f"  State:         {job.state.value}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")

### 1-B: Monitor Progress

In [ ]:
import time
from datetime import datetime

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
    JobStatus.PARTIALLY_FAILED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


completed_job = wait_for_job(client, job.id)

### 1-C: View Results

In [ ]:
def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")


display_job_results(completed_job)

---

## Evaluation 2: Multi-Benchmark Korean Evaluation

Submit multiple Korean benchmarks in a single request. EvalHub orchestrates them concurrently and tracks all results under one MLflow experiment.

In [ ]:
multi_request = JobSubmissionRequest(
    name="korean-multi-benchmark",
    description="Comprehensive Korean LLM evaluation: KMMLU + KoBEST + ARC",
    tags=["korean", "comprehensive", "multi-benchmark"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="kobest_wic",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="arc_easy",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-comprehensive-eval",
        tags=[
            ExperimentTag(key="model_family", value="gemma-4"),
            ExperimentTag(key="language", value="korean,english"),
            ExperimentTag(key="evaluation_type", value="comprehensive"),
        ],
    ),
)

print("Multi-Benchmark Request:")
print(f"  Name:       {multi_request.name}")
print(f"  Model:      {multi_request.model.name}")
print(f"  Benchmarks: {[b.id for b in multi_request.benchmarks]}")
print(f"  Experiment: {multi_request.experiment.name}")
print(f"\nUncomment the next cell to submit.")

In [ ]:
# Uncomment to submit the multi-benchmark evaluation:

# multi_job = client.jobs.submit(multi_request)
# print(f"Job submitted: {multi_job.id}")
# completed_multi = wait_for_job(client, multi_job.id)
# display_job_results(completed_multi)

---

## Evaluation 3: Few-Shot Comparison

Compare 0-shot vs 5-shot performance on the same benchmark. Each evaluation is tracked as a separate MLflow run within the same experiment for easy comparison.

In [ ]:
fewshot_configs = [
    {"name": "kmmlu-law-0shot", "num_fewshot": 0},
    {"name": "kmmlu-law-5shot", "num_fewshot": 5},
]

fewshot_jobs = []
for config in fewshot_configs:
    request = JobSubmissionRequest(
        name=config["name"],
        description=f"KMMLU Law {config['num_fewshot']}-shot evaluation",
        tags=["korean", "kmmlu", "fewshot-comparison"],
        model=model,
        benchmarks=[
            BenchmarkConfig(
                id="kmmlu_direct_law",
                provider_id="lm_evaluation_harness",
                parameters={
                    "num_fewshot": config["num_fewshot"],
                    "limit": LIMIT,
                },
            ),
        ],
        experiment=ExperimentConfig(
            name="kmmlu-fewshot-comparison",
            tags=[
                ExperimentTag(key="model_family", value="gemma-4"),
                ExperimentTag(key="comparison_type", value="fewshot"),
                ExperimentTag(key="num_fewshot", value=str(config["num_fewshot"])),
            ],
        ),
    )
    fewshot_jobs.append(request)
    print(f"Prepared: {config['name']} (num_fewshot={config['num_fewshot']})")

print(f"\n{len(fewshot_jobs)} jobs ready. Uncomment below to submit.")

In [ ]:
# Uncomment to submit few-shot comparison:

# submitted = []
# for req in fewshot_jobs:
#     j = client.jobs.submit(req)
#     print(f"Submitted: {j.name} -> {j.id}")
#     submitted.append(j)
#
# for j in submitted:
#     result = wait_for_job(client, j.id)
#     display_job_results(result)

---

## Step 4: Job Management

List, inspect, and manage all evaluation jobs.

### List All Jobs

In [ ]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {jobs_list.total_count}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':30s}  {'Experiment':25s}  Benchmarks")
print("-" * 90)
for j in jobs_list.items:
    state = j.effective_state.value
    exp = j.experiment.name if j.experiment else "N/A"
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:30]:30s}  {exp[:25]:25s}  {bms}")

### Inspect a Specific Job

Replace `JOB_ID` with the job you want to inspect.

In [ ]:
# Replace with actual job ID:
# JOB_ID = "your-job-id-here"
# inspected = client.jobs.get(JOB_ID)
# display_job_results(inspected)

---

## Step 5: Results Comparison

Compare results across multiple evaluation jobs. Collect metrics from completed jobs and display as a comparison table.

In [ ]:
def collect_results_table(client, job_ids=None):
    """Collect benchmark metrics from multiple jobs into a comparison dict.

    Returns: {benchmark_id: {job_name: {metric: value}}}
    """
    if job_ids is None:
        jobs_list = client.jobs.list()
        jobs = [
            j for j in jobs_list.items
            if j.effective_state == JobStatus.COMPLETED
        ]
    else:
        jobs = [client.jobs.get(jid) for jid in job_ids]

    table = {}
    for j in jobs:
        if not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics

    return table


comparison = collect_results_table(client)

if comparison:
    print("Results Comparison")
    print("=" * 70)
    for benchmark_id, job_results in comparison.items():
        print(f"\n  Benchmark: {benchmark_id}")
        print(f"  {'-' * 60}")
        for job_name, metrics in job_results.items():
            print(f"    {job_name}:")
            for metric, value in metrics.items():
                if isinstance(value, float):
                    print(f"      {metric:30s} = {value:.4f}")
                else:
                    print(f"      {metric:30s} = {value}")
else:
    print("No completed jobs found. Submit and complete evaluations first.")

### Comparison Table with pandas

In [ ]:
try:
    import pandas as pd

    rows = []
    for benchmark_id, job_results in comparison.items():
        for job_name, metrics in job_results.items():
            for metric, value in metrics.items():
                if isinstance(value, (int, float)):
                    rows.append({
                        "benchmark": benchmark_id,
                        "job": job_name,
                        "metric": metric,
                        "value": value,
                    })

    if rows:
        df = pd.DataFrame(rows)
        pivot = df.pivot_table(
            index=["benchmark", "metric"],
            columns="job",
            values="value",
        )
        display(pivot.style.format("{:.4f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No numeric results to display.")

except ImportError:
    print("pandas not available. Install with: pip install pandas")

---

## Step 6: MLflow Integration

Access evaluation results directly from MLflow for advanced analysis, comparison, and visualization.

In [ ]:
try:
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

    experiments = mlflow.search_experiments()
    print(f"\nMLflow Experiments ({len(experiments)}):")
    print("=" * 70)
    for exp in experiments:
        print(f"  [{exp.experiment_id:>4s}] {exp.name}")

except ImportError:
    print("mlflow not installed. Install with: pip install mlflow")
    print("MLflow integration is optional - results are still available via the EvalHub API.")

### Query MLflow Runs

Search for evaluation runs within a specific experiment.

In [ ]:
try:
    import mlflow

    EXPERIMENT_NAME = "korean-kmmlu-law"  # Change to your experiment name

    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp:
        runs = mlflow.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=["start_time DESC"],
            max_results=10,
        )

        if not runs.empty:
            print(f"Recent runs in '{EXPERIMENT_NAME}':")
            metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
            param_cols = [c for c in runs.columns if c.startswith("params.")]
            display_cols = ["run_id", "status", "start_time"] + metric_cols[:5]
            display(runs[display_cols].head(10))
        else:
            print(f"No runs found in experiment '{EXPERIMENT_NAME}'.")
    else:
        print(f"Experiment '{EXPERIMENT_NAME}' not found.")
        print("Available experiments:")
        for exp in mlflow.search_experiments():
            print(f"  - {exp.name}")

except ImportError:
    print("mlflow not installed. Skipping MLflow queries.")
except Exception as e:
    print(f"MLflow query failed: {e}")
    print("Make sure the MLflow server is accessible.")

### Visualize MLflow Metrics

In [ ]:
try:
    import mlflow
    import matplotlib.pyplot as plt

    EXPERIMENT_NAME = "korean-comprehensive-eval"  # Change as needed

    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp:
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        metric_cols = [c for c in runs.columns if c.startswith("metrics.")]

        if not runs.empty and metric_cols:
            metrics_df = runs[metric_cols].dropna(axis=1, how="all")
            metrics_df.columns = [c.replace("metrics.", "") for c in metrics_df.columns]

            fig, ax = plt.subplots(figsize=(12, 6))
            metrics_df.plot(kind="bar", ax=ax)
            ax.set_title(f"Benchmark Metrics - {EXPERIMENT_NAME}")
            ax.set_ylabel("Score")
            ax.set_xlabel("Run")
            ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            plt.tight_layout()
            plt.show()
        else:
            print("No metric data to visualize.")
    else:
        print(f"Experiment '{EXPERIMENT_NAME}' not found.")

except ImportError:
    print("mlflow or matplotlib not installed. Skipping visualization.")
except Exception as e:
    print(f"Visualization failed: {e}")

---

## Step 7: Export Results

### Export to Markdown

In [ ]:
def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

### Save Results to JSON

In [ ]:
import json
from pathlib import Path

output_dir = Path("../results/evalhub")
output_dir.mkdir(parents=True, exist_ok=True)

jobs_list = client.jobs.list()
for j in jobs_list.items:
    if j.effective_state != JobStatus.COMPLETED or not j.results:
        continue

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = output_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")

print(f"\nResults saved to {output_dir.resolve()}")

---

## Summary

This notebook demonstrated the full EvalHub evaluation workflow:

1. **Single benchmark** evaluation with MLflow tracking
2. **Multi-benchmark** evaluation in a single request
3. **Few-shot comparison** tracked under one MLflow experiment
4. **Job management** -- list, inspect, compare
5. **Results analysis** -- comparison tables with pandas
6. **MLflow integration** -- query runs, visualize metrics
7. **Export** -- Markdown and JSON output

### EvalHub vs LMEvalJob: When to Use Which

| Use Case | Recommended |
|----------|-------------|
| Quick single-task smoke test | **Phase 1** (1_builtin_tasks) |
| Full benchmark suite (KMMLU 45, CLIcK 11) | **Phase 2** (2_custom_tasks) |
| Local result analysis from LMEvalJob | **Phase 3** (3_local_benchmark) |
| Multi-framework, multi-benchmark + MLflow | **Phase 4** (this notebook) |
| CI/CD integration | **Phase 4** (EvalHub REST API) |
| Model comparison across versions | **Phase 4** (MLflow experiment tracking) |

### Useful Links

- [EvalHub API Docs](https://eval-hub.github.io/eval-hub/) -- Full endpoint reference
- [EvalHub SDK](https://github.com/eval-hub/eval-hub-sdk) -- Python client and adapter SDK
- [MLflow Integration](https://github.com/eval-hub/eval-hub/blob/main/MLFLOW.md) -- Experiment tracking guide
- [evaluate-llm-on-korean-dataset](https://github.com/hyogrin/evaluate-llm-on-korean-dataset) -- Korean LLM benchmark results